# Assignment 2.6 — Flow Matching from Scratch

Dataset: **2D 8-mode Gaussian mixture**

- `x0` = data
- `x1` = Gaussian noise
- `x_t = (1-t)x0 + t x1`
- target velocity: `v = x1 - x0`

Sampling starts from noise at `t=1` and uses Euler integration backward to `t=0`.

PyTorch only. No bonus.


In [ ]:
# Cell 1 — Kaggle P100 setup
# Run once. If torch is changed, restart the session and Run All.

import sys
import subprocess
import importlib.metadata as metadata

current = metadata.version("torch")
print("Current torch:", current)

if not current.startswith("2.7.1"):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q",
        "torch==2.7.1",
        "--index-url",
        "https://download.pytorch.org/whl/cu126"
    ])
    print("Installed torch 2.7.1 + cu126.")
    print("Restart Session, then Run All.")
else:
    print("P100-compatible PyTorch is already installed.")


In [ ]:
# Cell 2 — Imports and device
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    print("Supported arch:", torch.cuda.get_arch_list())

    assert "sm_60" in torch.cuda.get_arch_list(), (
        "P100 needs sm_60. Run Cell 1, restart session, then Run All."
    )


In [ ]:
# Cell 3 — Hyperparameters
batch_size = 512
train_steps = 5000
learning_rate = 1e-3

radius = 4.0
data_std = 0.25


In [ ]:
# Cell 4 — 8-mode Gaussian mixture
def sample_data(n):
    angles = torch.arange(8, device=device) * (2 * math.pi / 8)

    centers = torch.stack([
        radius * torch.cos(angles),
        radius * torch.sin(angles)
    ], dim=1)

    labels = torch.randint(0, 8, (n,), device=device)

    x0 = centers[labels] + data_std * torch.randn(
        n, 2, device=device
    )

    return x0


real = sample_data(3000).cpu()

plt.figure(figsize=(5, 5))
plt.scatter(real[:, 0], real[:, 1], s=5, alpha=0.5)
plt.axis("equal")
plt.title("8-mode Gaussian Mixture")
plt.show()


In [ ]:
# Cell 5 — VelocityNet(x_t, t) -> v
class VelocityNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(3, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x, t):
        inp = torch.cat([x, t], dim=1)
        return self.net(inp)


model = VelocityNet().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

print(model)


In [ ]:
# Cell 6 — Flow Matching training
loss_history = []

for step in range(1, train_steps + 1):
    # x0 = data
    x0 = sample_data(batch_size)

    # x1 = noise
    x1 = torch.randn(batch_size, 2, device=device)

    # t ~ Uniform(0,1)
    t = torch.rand(batch_size, 1, device=device)

    # Straight interpolation
    xt = (1 - t) * x0 + t * x1

    # Exact velocity of this path
    target_v = x1 - x0

    pred_v = model(xt, t)

    loss = F.mse_loss(pred_v, target_v)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if step % 500 == 0:
        print(
            f"Step [{step:04d}/{train_steps}] "
            f"Loss: {loss.item():.6f}"
        )


In [ ]:
# Cell 7 — Training loss
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Training step")
plt.ylabel("FM MSE Loss")
plt.title("Flow Matching Training Loss")
plt.grid(True)
plt.show()


In [ ]:
# Cell 8 — Euler sampler
@torch.no_grad()
def sample_euler(model, x_start, n_steps):
    model.eval()

    x = x_start.clone()
    trajectory = [x.cpu()]

    # Integrate from t=1 -> t=0
    dt = -1.0 / n_steps

    for i in range(n_steps):
        t_value = 1.0 - i / n_steps

        t = torch.full(
            (x.size(0), 1),
            t_value,
            device=device
        )

        v = model(x, t)
        x = x + dt * v

        trajectory.append(x.cpu())

    return x.cpu(), trajectory


In [ ]:
# Cell 9 — Sample with 1, 5, 10, 50 steps
n_samples = 2000

# Same starting noise for all step counts
x_noise = torch.randn(
    n_samples, 2,
    device=device
)

results = {}

for steps in [1, 5, 10, 50]:
    samples, trajectory = sample_euler(
        model,
        x_noise,
        steps
    )

    results[steps] = {
        "samples": samples,
        "trajectory": trajectory
    }

    print(f"Finished {steps} Euler step(s)")


In [ ]:
# Cell 10 — Generated samples
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, steps in zip(axes.flat, [1, 5, 10, 50]):
    samples = results[steps]["samples"]

    ax.scatter(
        samples[:, 0],
        samples[:, 1],
        s=5,
        alpha=0.5
    )

    ax.set_title(f"Euler Sampling — {steps} step(s)")
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()


In [ ]:
# Cell 11 — Trajectories
n_show = 20
background = sample_data(1500).cpu()

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, steps in zip(axes.flat, [1, 5, 10, 50]):
    traj = torch.stack(
        results[steps]["trajectory"]
    )

    ax.scatter(
        background[:, 0],
        background[:, 1],
        s=4,
        alpha=0.15
    )

    for i in range(n_show):
        ax.plot(
            traj[:, i, 0],
            traj[:, i, 1],
            marker="o",
            markersize=2
        )

    ax.set_title(f"Trajectories — {steps} step(s)")
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()
